# HDGPSO Benchmark

**HDGPSO** is a three-stage hybrid metaheuristic for hyperparameter optimization:
1. **DE** — Differential Evolution mutation/crossover/selection (exploration).
2. **GWO** — Grey Wolf leadership update using top-3 (α, β, δ) of the population.
3. **PSO** — Particle Swarm with personal-best, global-best, and linearly-decreasing inertia.

This notebook compares HDGPSO against:
- GridSearchCV (coarse)
- RandomSearch
- Bayesian (scikit-optimize)
- Optuna-TPE
- Plain DE (scipy)
- Plain PSO (pyswarms)

across multiple sklearn datasets × {RandomForest, GradientBoosting, XGBoost} × N seeds, and produces:
- A Demšar (2006) statistical analysis: Friedman test, Nemenyi post-hoc, Critical-Difference (CD) diagram.
- Effect sizes (Cliff's δ, median % improvement) and Wilcoxon paired tests.
- Bootstrap 95% CIs on per-tuner mean rank.
- Convergence curves and headline tables.
- An optional **budget sensitivity sweep** (budget ∈ {20, 40, 60, 100}) showing HDGPSO holds up across budgets.

Run cells in order. The benchmark cell is the slow one (~15-30 min for the sanity run, 1-3 hr for the full paper config, 4-12 hr for the budget sweep).

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '.')

from benchmark import run_benchmark, run_budget_sweep, aggregate, rank_table
from plots import convergence_plot, convergence_grid, best_loss_bar, mean_rank_bar, wins_table
from stats import (
    friedman_test, nemenyi_matrix, cd_diagram, demsar_report,
    hdgpso_vs_baselines_table, bootstrap_rank_ci, cliffs_delta, critical_difference,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Sanity run

Quick smoke on one dataset × one model × two seeds. Confirms imports work and HDGPSO runs end-to-end before the full benchmark.

In [ ]:
summary_smoke, history_smoke = run_benchmark(
    budget=30,
    seeds=[0, 1],
    datasets=['breast_cancer'],
    models=['RandomForest'],
    cv=3,
    population_size=6,
    include_openml=False,
    out_dir='results_smoke',
    verbose=True,
)
summary_smoke

In [ ]:
rank_table(summary_smoke)

In [ ]:
convergence_plot(history_smoke, 'breast_cancer', 'RandomForest')

## 2. Full benchmark

Defaults are the paper-quality run.

In [ ]:
CONFIG = dict(
    budget=60,
    seeds=[0, 1, 2],
    cv=3,
    population_size=10,
    include_openml=True,
    out_dir='results',
)

t0 = time.time()
summary, history = run_benchmark(**CONFIG, verbose=True)
print(f'\nTotal benchmark time: {(time.time()-t0)/60:.1f} min')
summary.head()

## 3. Headline numbers

In [ ]:
print('--- Mean rank (lower = better) ---')
rank_table(summary)

In [ ]:
print('--- Wins per tuner ---')
wins_table(summary)

In [ ]:
# Paper-friendly pivot: rows = (dataset, model), cols = tuner, vals = mean ± std
agg = aggregate(summary)
def fmt(row): return f"{row['mean']:.4f} ± {row['std']:.4f}"
agg['cell'] = agg.apply(fmt, axis=1)
agg.pivot_table(index=['dataset','model'], columns='tuner', values='cell', aggfunc='first')

## 4. Demšar (2006) statistical analysis

Publication-grade methodology for comparing multiple algorithms over multiple datasets:

**4.1 Friedman test** — global null hypothesis: "all tuners are equivalent across all cells." Must reject before claiming any tuner is better.

In [ ]:
friedman_test(summary)

**4.2 Mean rank with bootstrap 95% CI** — resamples (dataset, model, seed) cells with replacement to show rank stability.

In [ ]:
bootstrap_rank_ci(summary, n_boot=2000, seed=0)

**4.3 HDGPSO vs each baseline** — rank delta, Wilcoxon p-value, Cliff's δ effect size, Nemenyi-CD significance flag.

In [ ]:
hdgpso_vs_baselines_table(summary, target='HDGPSO')

**4.4 Critical Difference (CD) diagram** — the single figure that summarizes the whole comparison. Tuners are placed on a number line by mean rank (best = left). Horizontal bars connect tuners whose rank difference is **less than the critical difference** (i.e., not statistically distinguishable). Tuners with bars between them are statistically tied; tuners without a connecting bar differ significantly.

In [ ]:
fig = cd_diagram(summary, save_path='fig_cd_diagram.png',
                 title=f'Critical Difference diagram (α=0.05, n={summary.pivot_table(index=["dataset","model","seed"], columns="tuner", values="best_loss").dropna().shape[0]} cells)')
plt.show()

**4.5 Full Demšar report** — same numbers printed inline for copy-paste into the paper.

In [ ]:
_ = demsar_report(summary, target='HDGPSO')

## 5. Other figures

In [ ]:
fig = mean_rank_bar(summary, save_path='fig_mean_rank.png')
plt.show()

In [ ]:
fig = convergence_grid(history, save_path='fig_convergence.png')
plt.show()

## 6. Budget sensitivity sweep

Strongest defence against the "works only at one tuned budget" reviewer criticism. Runs the entire benchmark at four budgets and shows how each tuner's mean rank evolves.

**Warning: 4× slower than the main benchmark.** Skip this cell if iterating; run once for the final paper figure.

In [ ]:
SWEEP_CONFIG = dict(
    budgets=[20, 40, 60, 100],
    seeds=[0, 1, 2],
    cv=3,
    population_size=10,
    include_openml=True,
    out_dir='results_budget_sweep',
)

t0 = time.time()
summary_sweep, history_sweep = run_budget_sweep(**SWEEP_CONFIG, verbose=True)
print(f'\nBudget sweep total time: {(time.time()-t0)/60:.1f} min')

In [ ]:
# Rank vs budget per tuner
import seaborn as sns  # optional; if missing, fall back below
rank_by_budget = []
for b in sorted(summary_sweep['budget'].unique()):
    s = summary_sweep[summary_sweep['budget'] == b].copy()
    s['rank'] = s.groupby(['dataset','model','seed'])['best_loss'].rank()
    rb = s.groupby('tuner')['rank'].mean().reset_index()
    rb['budget'] = b
    rank_by_budget.append(rb)
rank_by_budget = pd.concat(rank_by_budget, ignore_index=True)
rank_by_budget

In [ ]:
from plots import TUNER_COLORS
fig, ax = plt.subplots(figsize=(7, 4.5))
for t in rank_by_budget['tuner'].unique():
    sub = rank_by_budget[rank_by_budget['tuner'] == t]
    ax.plot(sub['budget'], sub['rank'], marker='o', color=TUNER_COLORS.get(t, '#444'), label=t, linewidth=1.8)
ax.invert_yaxis()  # rank 1 is best, so put 1 at the top
ax.set_xlabel('Function-evaluation budget')
ax.set_ylabel('Mean rank across all cells (lower = better)')
ax.set_title('Budget sensitivity: rank vs evaluation budget')
ax.legend(fontsize=8, loc='best')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('fig_budget_sweep.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# CD diagram at each budget — supplementary material
fig, axes = plt.subplots(1, len(SWEEP_CONFIG['budgets']), figsize=(6*len(SWEEP_CONFIG['budgets']), 4))
for ax, b in zip(axes, sorted(summary_sweep['budget'].unique())):
    sub = summary_sweep[summary_sweep['budget'] == b]
    cd_diagram(sub, ax=ax, title=f'Budget = {b}')
fig.tight_layout()
fig.savefig('fig_cd_by_budget.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Paper claims checklist

After the full benchmark runs, the following claims should be checked and reported:

1. **Friedman test rejects H0** at α=0.05 (cell 4.1).
2. **HDGPSO has the lowest mean rank** (cells 4.2 / 4.4).
3. **HDGPSO is Nemenyi-significantly better** than at least one named baseline (cell 4.3, `nemenyi_significant=True` column).
4. **Wilcoxon p < 0.05** for HDGPSO vs each underperforming baseline (cell 4.3).
5. **Cliff's δ > 0.147** (non-negligible effect size) vs underperforming baselines (cell 4.3).
6. **HDGPSO's rank stays low across budgets** in the sweep (cell 6).

If any of these fail on the full benchmark, the paper claim must be softened — e.g. "competitive with Bayes/Optuna" rather than "superior to all baselines."